# N032 · 前缀和与哈希计数

**目标：** 将区间条件改写为前缀值之间的关系。

**先修：** N019, N031。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 前缀差；频次与最早位置；余数类；初始前缀0。

**配套讲解来源：** [同名逐章意见](../../comment/032_prefix_hash_counting.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章解决两个"数连续子数组"的问题。

**问题 A（和为 k 的子数组个数）**：给你整数数组 nums 和整数 k，问有多少个**非空连续子数组**的元素和恰好等于 k。数组里可以有负数，所以滑动窗口不好使，得换思路。具体例子：输入 `nums=[1,1,1]`、`k=2`，输出 `2`——两个合法子数组是 `[1,1]`（下标 0-1）和 `[1,1]`（下标 1-2），位置不同算两个；而 `[1,1,1]` 和是 3 不算，单个 `[1]` 和是 1 也不算。

**问题 B（长度至少为 2 且和为 k 的倍数）**：给你数组 nums 和整数 k，判断是否存在一个**长度至少为 2** 的连续子数组，其元素和是 k 的倍数。具体例子：输入 `nums=[23,2,4,6,7]`、`k=6`，输出 `True`。为什么？因为子数组 `[2,4]` 的和是 6，长度是 2，正好是 6 的 1 倍。再如 `nums=[6]`、`k=6`：虽然 6 是 6 的倍数，但长度只有 1，不满足"至少 2"，所以答案是 `False`。

两个问题的共同武器是：**把"区间的条件"翻译成"两个前缀值之间的关系"**，然后用哈希表快速查"以前有没有出现过满足关系的前缀"。这一步翻译是前缀和章节（N031）与哈希章节（N019）的合流点。

## 2. 基础知识：先读懂这些词

- **前缀和（P）**：P[i] 表示前 i 个元素之和，P[0]=0。任意区间 [l,r) 的和等于 P[r]−P[l]。这一章的一切都建立在这个等式上。
- **前缀差**：区间和为 k，把它代入上式移项就得到 `P[r] = P[l] + k`。也就是说："存在一段和为 k 的区间"等价于"存在两个前缀，它们相差 k"。问题从"找区间"变成"找一对前缀值"，这一步是全章的枢纽。
- **哈希计数（频次字典）**：用一个字典记录"每个前缀和出现过的次数"。扫到当前位置得到前缀 P 时，查字典里 `P−k` 出现过几次——每出现一次，就对应一个合法区间的左端。字典把"查找"降到平均 O(1)，这是把整题做到 O(n) 的关键。
- **初始前缀 0（counts 里预放 {0:1}）**：空前缀 P[0]=0 也是一个真实存在的前缀。如果忘了把它放进字典，从数组开头开始的区间（l=0）就永远配不上对，会漏解。比如 [1,1,1] 找 k=2，扫到第二个 1 时前缀是 2，要找 2−2=0，这个 0 就来自初始前缀。这是本题最著名的坑。
- **余数类**：两个数 mod k 的余数相同，当且仅当它们的差是 k 的倍数。所以"区间和是 k 的倍数"等价于"区间两端的前缀和 mod k 同余"。这样我们不用关心具体差值，只需按余数分组。
- **最早位置（first 字典存每个余数第一次出现的下标）**：判断"长度至少 2"时，对同一个余数我们只需要记住它**最早**出现的下标——因为当前下标减去最早下标得到的间距最大，如果连最大间距都不足 2，那更晚出现的同余前缀更不够。注意 `first={0:-1}`：余数 0 的"最早出现"设定在下标 −1（代表空前缀），这样从开头起算的区间长度才是真实长度。
- **defaultdict**：查不存在的 key 返回默认值 0 的字典，用作计数器可以少写一行判断。它还能在构造时预置初始内容（如 `{0:1}`），"默认工厂 + 初始字典"一步到位。

## 3. 思路推导：从小例子开始

问题 A 的推导：

- **Step 1**：一边扫数组一边维护当前前缀和 `prefix`。
- **Step 2**：在把当前位置计入字典**之前**，先查字典里有多少个 `prefix−k`。此时字典里存的都是严格更早的前缀，所以配出来的区间长度至少为 1（天然排除空区间）。
- **Step 3**：查询完再把当前 `prefix` 计入字典。这个"先查后存"的顺序保证了不会用自己配自己（那样会算出长度 0 的区间）。

手算演示（`a=[1,-1,0]`、k=0，正是 “运行示例”部分 小实例的数据）：

| i | 读入 | 前缀和 | 字典里 0 的个数（新增零和区间） | 累计 |
|---|---|---|---|---|
| 0 | 1 | 1 | 0 | 0 |
| 1 | -1 | 0 | 1（配出 [1,-1]） | 1 |
| 2 | 0 | 0 | 2（配出 [1,-1,0] 和 [0]） | 3 |

最终答案 3。这张表就是 notebook “运行示例”部分 用 `show_table` 打印的内容（列：i、前缀和、新增零和区间、累计）。核对一下：和为 0 的子数组确实是 [1,−1]、[1,−1,0]、[0] 这三个。

再看 k≠0 的标准样例（`nums=[1,1,1]`、k=2，“自动测试”部分 第一条断言）：

| i | 读入 | prefix | 查 counts[prefix−2] | 登记 |
|---|---|---|---|---|
| 0 | 1 | 1 | counts[−1]=0 | counts={0:1, 1:1} |
| 1 | 1 | 2 | counts[0]=1 → 新增 1 | counts={0:1, 1:1, 2:1} |
| 2 | 1 | 3 | counts[1]=1 → 新增 1 | counts={0:1, 1:1, 2:1, 3:1} |

两步命中分别对应子数组 [1,1]（下标 0–1）和 [1,1]（下标 1–2），答案 2。i=1 那次查到的 0 就是预置的初始前缀——如果字典忘了预置 {0:1}，这一步会查到 0，答案错成 1。

问题 B 的推导：

- **Step 1**：维护前缀和，每步算余数 `prefix % abs(k)`（对 k 取绝对值是为了兼容负 k；k=0 时退化为直接比较前缀和本身，代表"区间和必须恰为 0"）。
- **Step 2**：如果这个余数以前出现过，算当前下标与**最早下标**的差；差 ≥2 就返回 True。没出现过就记下当前下标。
- **Step 3**：扫完都没遇到，返回 False。

手算演示（`nums=[23,2,4,6,7]`，k=6）：前缀和依次 23、25、29、35、42，mod 6 余数依次 5、1、5、5、0。first 初始 {0:−1}。逐步走：

| i | 读入 | 前缀和 | 余数 (mod 6) | first 里的动作 | 结果 |
|---|---|---|---|---|---|
| 0 | 23 | 23 | 5 | 5 没见过，first[5]=0 | 继续 |
| 1 | 2 | 25 | 1 | 1 没见过，first[1]=1 | 继续 |
| 2 | 4 | 29 | 5 | 5 见过（下标 0），间距 2−0=2 ≥ 2 | **返回 True** |

这轮命中的配对是"i=0 时存下的前缀 23"和"i=2 时的当前前缀 29"：23 mod 6 = 29 mod 6 = 5，差 29−23=6 是 6 的倍数。这对前缀夹住的子数组是下标 1、2 两个元素 [2,4]，和恰为 6。间距公式 `i−first[key]=2−0=2` 数的正是这个子数组的元素个数——存的是"上一次出现时的循环下标"，当前下标减它就得到中间夹着的元素数。

## 4. 核心代码：subarray_sum

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def subarray_sum(nums, k):
    counts = defaultdict(int, {0: 1})
    prefix = answer = 0
    for x in nums:
        prefix += x
        answer += counts[prefix - k]
        counts[prefix] += 1
    return answer
```

### 逐段读懂代码

先看 `subarray_sum`，对应实现如下：

```python
from collections import defaultdict

def subarray_sum(nums, k):
    counts = defaultdict(int, {0: 1})
    prefix = answer = 0
    for x in nums:
        prefix += x
        answer += counts[prefix - k]
        counts[prefix] += 1
    return answer
```

- `counts=defaultdict(int,{0:1})`：建计数字典并**预置前缀 0 出现过一次**。`defaultdict(int, {...})` 表示"以 int 为默认工厂、初始内容为 {0:1}"。这个 0 代表空的前缀 P[0]，漏掉它就会漏掉所有从下标 0 开始的区间。
  - 拿 [1,1,1]、k=2 检验：扫到第二个元素时 prefix=2，要找 2−2=0；字典里若没有预置的 0，这次命中就丢了，答案会从 2 错成 1。
- `prefix=answer=0`：prefix 是滚动前缀和，answer 是累计答案。
- `prefix+=x`：把当前元素累进前缀。
- `answer+=counts[prefix-k]`：查"比我早、值恰好是 prefix−k 的前缀有几个"。每一次命中对应一个和为 k 的区间（因为 prefix − (prefix−k) = k）。用 defaultdict 的好处：key 不存在时返回 0，不用先判断在不在。
  - 负数 key 完全合法：prefix−k 可能是负数（比如前缀 1 减 k=2 得 −1），字典照样按值查，这正是该解法天然兼容负数数组的原因。
- `counts[prefix]+=1`：查完才登记自己。先查后存的顺序保证字典里只有严格更早的前缀，不会把自己算进去。
- `return answer`：所有右端点的新增数之和就是总个数。

再看 `check_subarray_sum`，对应实现如下：

```python
def check_subarray_sum(nums, k):
    first = {0: -1}
    prefix = 0
    for i, x in enumerate(nums):
        prefix += x
        key = prefix % abs(k) if k else prefix
        if key in first:
            if i - first[key] >= 2:
                return True
        else:
            first[key] = i
    return False
```

- `first={0:-1}`：普通字典，key 是余数（或前缀和本身），value 是该 key **第一次**出现的下标。余数 0 预置在下标 −1，代表"数组开始之前"的空前缀，这样 l=0 的区间长度才能算对。拿 [6]、k=6 检验：i=0 时 prefix=6 余 0，first[0]=−1，间距 0−(−1)=1，不足 2，返回 False——正是 “自动测试”部分 那条"单元素不算"断言的行为。
- `prefix+=x`：滚动前缀和。
- `key=prefix%abs(k) if k else prefix`：这是 Python 的条件表达式（三元写法）。k 非 0 时取余数（对 |k| 取余，兼容负 k）；k 为 0 时直接用前缀和本身当 key，此时判据退化为"两前缀相等"，即区间和为 0。
- `if key in first:`：这个余数以前出现过。两个同余前缀之差是 k 的倍数，正好是某个区间的和。
- `if i-first[key]>=2: return True`：区间长度（元素个数）是 `i−first[key]`，这里下标都是从 0 数的，元素个数恰等于两下标之差；要求至少 2 个元素。用最早下标保证间距取到最大，若最大间距都不足 2 就真的不存在。
- `else: first[key]=i`：只在第一次出现时记录。如果已经存在但间距不足 2，**不覆盖**——覆盖成更晚的下标只会让以后的间距更小，保留最早的才有机会凑足长度。
  - 注意和问题 A 的区别：那边字典存"次数"（要数个数），这边字典存"最早下标"（要判长度、求最长）。同一套"前缀 + 哈希"骨架，字典里放什么取决于题目问什么。
- `return False`：扫完全程没找到满足条件的配对。

## 5. 分段实现：按顺序运行

**本章接口：** `subarray_sum(nums, k)`、`check_subarray_sum(nums, k)`

### 导入本章使用的库

In [1]:
from collections import defaultdict

### subarray_sum

In [2]:
def subarray_sum(nums, k):
    counts = defaultdict(int, {0: 1})
    prefix = answer = 0
    for x in nums:
        prefix += x
        answer += counts[prefix - k]
        counts[prefix] += 1
    return answer

### check_subarray_sum

In [3]:
def check_subarray_sum(nums, k):
    first = {0: -1}
    prefix = 0
    for i, x in enumerate(nums):
        prefix += x
        key = prefix % abs(k) if k else prefix
        if key in first:
            if i - first[key] >= 2:
                return True
        else:
            first[key] = i
    return False

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [4]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[1,-1,0]; prefix=0; seen={0:1}; rows=[]; total=0
for i,x in enumerate(a):
    prefix+=x; increment=seen.get(prefix,0); total+=increment
    rows.append((i,prefix,increment,total)); seen[prefix]=seen.get(prefix,0)+1
show_table(['i','前缀和','新增零和区间','累计'],rows)

i,前缀和,新增零和区间,累计
0,1,0,0
1,0,1,1
2,0,2,3


## 7. 正确性、适用条件与复杂度

查询时字典只包含严格更早的前缀，因此不会把长度0区间计入。相同余数等价于差是k的倍数；最早前缀给出最大可能间距。

**代价：** 平均 O(n) 时间、O(n) 空间；余数版非零k时不同余数至多 |k| 个。

### 展开理解

**为什么对（问题 A）**：每个和为 k 的区间 [l,r) 都唯一对应一对前缀 (P[l], P[r])，满足 P[r]−P[l]=k。我们从左到右扫描，站在 r 的角度看，合法的 l 就是那些"P[l] = P[r]−k"的更早位置，字典一步查出个数。反过来，字典里登记的每个前缀都真实对应某个下标，配出的区间都真实存在且和为 k，所以不重不漏——每个区间恰好被它的右端点数了一次。空区间之所以不会被数进去，是因为查询发生在登记自己之前，自己永远配不到自己。

**为什么对（问题 B）**：两个前缀 mod k 余数相同，它们的差就是 k 的整数倍；而前缀之差恰是中间那段区间的和。长度方面，同一个余数可能出现多次，但能与之配出最长区间的一定是最早出现的那次，所以我们只存最早下标；只要存在任何一对间距 ≥2 的同余前缀，最早的那对必然满足（间距只会更大或相等）。

**复杂度**：两个函数都是单趟扫描，每步做常数次字典操作，平均 O(n) 时间。具体数字：数组十万长时约十万次哈希查询加十万次更新，毫秒级；对比暴力枚举所有 O(n^2) 个子数组再各自求和（还要再乘个 n 如果不滚动），是百亿量级，没法比。空间上问题 A 的字典最多存 n+1 个不同的前缀值，O(n)；问题 B 在 k 非 0 时最多只有 |k| 种不同余数，字典大小不超过 |k|+1，但 k 可能很大所以一般仍写 O(n) 上界。

还有一层容易忽略的对比：为什么不用 N030 的滑动窗口做问题 A？因为窗口收缩依赖"删左边让条件单调恢复"，而和为 k 的区间在有负数时左端点不连续，窗口会漏解。前缀 + 哈希不关心数组里有没有负数——P[r]−P[l]=k 这个等式对任何整数都成立，所以它是带负数版本的通用解。这也解释了本章先修为什么列着 N031：先把"前缀之差等于区间"这一步焊牢，哈希只是加速查找的工具。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的断言逐条解释：

- `assert subarray_sum([1,1,1],2)==2`：正常值测试。两个 `[1,1]`，我们在第一节手算过，验证主逻辑和"初始前缀 0"的预置是否正确。
- `assert subarray_sum([0,0,0],0)==6`：特殊值测试。全部为 0 找 k=0：非空子数组共 3+2+1=6 个，全部合法。这条考察同一个前缀值重复出现时计数是否正确累加（字典存"次数"而不是"是否出现过"）。
- `assert check_subarray_sum([23,2,4,6,7],6)`：正常值测试。LeetCode 523 样例，[2,4] 和为 6，返回 True，验证余数配对逻辑。
- `assert not check_subarray_sum([6],6)`：边界值测试。和是 6 的倍数但长度只有 1，必须返回 False。这条专门卡"忘了长度至少 2"的常见错误实现，也顺便检验 first[0]=−1 的设定（间距 0−(−1)=1 不足 2）。
- `assert check_subarray_sum([0,0],0)`：特殊值测试。k=0 的教学扩展分支：两个 0 组成长度 2、和为 0 的子数组。这条卡 `if k else prefix` 那个三元分支——如果 k=0 时还去模零，程序会直接崩溃。
- 最后的大循环：枚举长度 5、取值 {−1,0,1} 的全部数组（3^5=243 个）与 k∈{−2,−1,0,1,2}，和暴力对拍。拆开看两个参考实现：
  - 个数版：`sum(sum(a[l:r])==k for l in range(5) for r in range(l+1,6))` 枚举全部非空子数组（r 从 l+1 起），和恰好等于 k 就计 1。
  - 布尔版：`any((sum(a[l:r])%abs(k)==0 if k else sum(a[l:r])==0) for l in range(5) for r in range(l+2,6))` 枚举全部长度 ≥2 的子数组（r 从 l+2 起），k 非 0 时查"和是 |k| 的倍数"，k=0 时查"和恰为 0"。条件表达式放在循环里，两种判据共用一个骨架。
  - 负 k 用 `%abs(k)` 覆盖（余数只认 |k|），零元素和正负混合由取值集 {−1,0,1} 天然覆盖。243×5=1215 种组合全过，两个函数在小输入上的行为被完整钉死。

In [5]:
assert subarray_sum([1, 1, 1], 2) == 2

assert subarray_sum([0, 0, 0], 0) == 6

assert check_subarray_sum([23, 2, 4, 6, 7], 6)

assert not check_subarray_sum([6], 6)

assert check_subarray_sum([0, 0], 0)

from itertools import product

for a in product([-1, 0, 1], repeat=5):
    for k in range(-2, 3):
        assert subarray_sum(a, k) == sum((sum(a[l:r]) == k for l in range(5) for r in range(l + 1, 6)))
        ref = any((sum(a[l:r]) % abs(k) == 0 if k else sum(a[l:r]) == 0 for l in range(5) for r in range(l + 2, 6)))
        assert check_subarray_sum(a, k) == ref

print('N032: 所有本章断言通过')

N032: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 分别求个数和最长长度。

**练习 2：** 给出负数使普通和窗口失败的例子。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（分别求个数和最长长度）**：提示：求个数就是本章 `subarray_sum`（字典存**次数**）。求最长长度时把字典改成存"该前缀值**第一次**出现的下标"（类似问题 B 的 first 字典），再遇到相同前缀值时不覆盖，用 `i − first[prefix−k]` 更新最大长度。注意两个方向都要保留初始前缀 0：个数版预置 {0:1}，长度版预置 {0:−1}（或 {0:0}，取决于你用左端下标还是元素个数计长度，先想清楚再写）。手算样例可用 [1,−1,0] 找 k=0：个数 3，最长长度 3。
- **练习 2（给出负数使普通和窗口失败的例子）**：提示：滑动窗口处理"和恰为 k"依赖"窗口变长和变大"的单调性，负数会破坏它。试 `nums=[1,-1,1]`、k=1：如果你在右端读入 −1 后发现和变小就收缩左端，可能把左边的 1 提前扔掉，而正确答案里 [1,−1,1] 整段和恰为 1。先手算列出全部 4 个和为 1 的子数组，再对照"窗口收缩一步就回不来"的位置，说明窗口漏解的原因是同一右端点的合法左端点不连续。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [6]:
# 完整实现：本单元格不依赖其他单元格。
from collections import defaultdict

def subarray_sum(nums, k):
    counts = defaultdict(int, {0: 1})
    prefix = answer = 0
    for x in nums:
        prefix += x
        answer += counts[prefix - k]
        counts[prefix] += 1
    return answer

def check_subarray_sum(nums, k):
    first = {0: -1}
    prefix = 0
    for i, x in enumerate(nums):
        prefix += x
        key = prefix % abs(k) if k else prefix
        if key in first:
            if i - first[key] >= 2:
                return True
        else:
            first[key] = i
    return False

# 示例与回归测试
assert subarray_sum([1, 1, 1], 2) == 2

assert subarray_sum([0, 0, 0], 0) == 6

assert check_subarray_sum([23, 2, 4, 6, 7], 6)

assert not check_subarray_sum([6], 6)

assert check_subarray_sum([0, 0], 0)

from itertools import product

for a in product([-1, 0, 1], repeat=5):
    for k in range(-2, 3):
        assert subarray_sum(a, k) == sum((sum(a[l:r]) == k for l in range(5) for r in range(l + 1, 6)))
        ref = any((sum(a[l:r]) % abs(k) == 0 if k else sum(a[l:r]) == 0 for l in range(5) for r in range(l + 2, 6)))
        assert check_subarray_sum(a, k) == ref

print('N032: 所有本章断言通过')

N032: 所有本章断言通过


## 11. 代表题与迁移

- **560. Subarray Sum Equals K**：这题就是 `subarray_sum` 的原题（和为 K 的子数组个数），练的是"前缀差 + 哈希计数 + 初始前缀 0"三件套。这题数组含负数，官方数据里滑动窗口解法会被卡，前缀哈希是标准解。
- **523. Continuous Subarray Sum**：这题就是 `check_subarray_sum` 的原题（长度 ≥2 且和为 n 的倍数），练的是"余数类分组 + 保存最早下标判长度"。这题最容易翻车的两处——余数 0 要预置 −1、长度不足 2 不能算——本章断言各有一条专门盯着。

**题面入口：** [560](https://leetcode.com/problems/subarray-sum-equals-k/)

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。